# Task 6: BERT Embeddings & Encoder

Combining token, segment, and position embeddings.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

## BERT Embeddings

BERT uses three types of embeddings:
1. **Token embeddings**: WordPiece tokens
2. **Segment embeddings**: [SEP] distinguishes sentences
3. **Position embeddings**: Positional information

In [ ]:
class BERTEmbeddings(nn.Module):
    """BERT embedding layer combining token, segment, and position embeddings."""
    
    def __init__(self, vocab_size, d_model, max_len=512, segment_vocab_size=2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.segment_embedding = nn.Embedding(segment_vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, token_ids, segment_ids=None):
        batch_size, seq_len = token_ids.shape
        
        # Position indices
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0).expand(batch_size, -1)
        
        # Token embeddings
        token_embeds = self.token_embedding(token_ids)
        
        # Segment embeddings (default to 0 if not provided)
        if segment_ids is None:
            segment_ids = torch.zeros_like(token_ids)
        segment_embeds = self.segment_embedding(segment_ids)
        
        # Position embeddings
        position_embeds = self.position_embedding(positions)
        
        # Combine and normalize
        embeddings = token_embeds + segment_embeds + position_embeds
        embeddings = self.norm(embeddings)
        embeddings = self.dropout(embeddings)
        
        return embeddings

# Test
vocab_size = 30000
d_model = 768
embeddings = BERTEmbeddings(vocab_size=vocab_size, d_model=d_model)

token_ids = torch.randint(0, vocab_size, (2, 128))  # batch=2, seq=128
segment_ids = torch.tensor([[0]*64 + [1]*64 for _ in range(2)])  # Two sentences

output = embeddings(token_ids, segment_ids)
print(f"Input: token_ids {token_ids.shape}, segment {segment_ids.shape}")
print(f"Output: {output.shape}")

## Complete BERT Encoder

In [ ]:
class BERTEncoder(nn.Module):
    """Complete BERT Encoder (simplified BERT-base)."""
    
    def __init__(self, vocab_size, d_model=768, num_heads=12, num_layers=12, 
                 d_ff=3072, max_len=512, dropout=0.1):
        super().__init__()
        
        # Embeddings
        self.embeddings = BERTEmbeddings(vocab_size, d_model, max_len)
        
        # Encoder layers
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.pooler = nn.Linear(d_model, d_model)
        self.pooler_activation = nn.Tanh()
        
    def forward(self, token_ids, segment_ids=None, attention_mask=None):
        # Create embeddings
        x = self.embeddings(token_ids, segment_ids)
        
        # Encoder
        for layer in self.encoder_layers:
            x = layer(x)
        
        # Pooler (take [CLS] token representation)
        pooled = self.pooler_activation(self.pooler(x[:, 0]))
        
        return x, pooled


# Import from previous task
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, embed_dim),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        attn, _ = self.self_attention(x, x, x)
        x = self.norm1(x + self.dropout(attn))
        x = self.norm2(x + self.ffn(x))
        return x

# Test BERT-base (smaller version for demo)
bert = BERTEncoder(vocab_size=30000, d_model=256, num_heads=8, num_layers=6, d_ff=1024)
token_ids = torch.randint(0, 30000, (2, 128))
sequence_output, pooled_output = bert(token_ids)

print(f"Sequence output: {sequence_output.shape}")  # (batch, seq_len, d_model)
print(f"Pooled output: {pooled_output.shape}")  # (batch, d_model)
print(f"Parameters: {sum(p.numel() for p in bert.parameters()):,}")

## Summary
- ✅ Token + Segment + Position embeddings
- ✅ Multiple encoder layers
- ✅ [CLS] token pooling

## Next: [Task 7: BERT Pretraining (MLM + NSP)](../task-07-bert-pretraining/overview.html)